# video-notes: GPU OCR benchmark on Colab

Runs `frames.py` on the four hand-labelled test videos with three OCR modes and scores each:

- **tesseract**: the default (CPU)
- **hybrid**: Tesseract, plus RapidOCR on the talking-head review tiles to rescue overlays
- **rapidocr**: RapidOCR for every frame (GPU)

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
**Kaggle:** Settings → Accelerator **GPU T4 x2** and **Internet on** (needs a phone-verified account), then Run all.
Takes about 45–60 minutes. Keep the tab open.

## 1. GPU

In [ ]:
import subprocess, torch
import shutil
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else "No GPU in this session (fine for hybrid mode)")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("CPU cores:", __import__("os").cpu_count())

## 2. Install the tools and the skill

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -U yt-dlp rapidocr onnxruntime imagehash "opencv-python>=4.8,<5"
!rm -rf video-notes && git clone -q -b gpu-ocr https://github.com/endgame302/video-notes.git
!python video-notes/skills/video-notes/scripts/check_deps.py

## 3. Test videos

Downloads from YouTube. If YouTube blocks Colab ("Sign in to confirm you're not a bot"), put each
video in your Google Drive as `MyDrive/vn-bench/<id>/video.mp4` and run the next cell too.

In [ ]:
import json, pathlib, shutil, subprocess
IDS = ["YlY2g2xrl6Q", "BxV14h0kFs0", "jnaljahKibQ", "nEaJMdM1djM"]
import os
# Colab works in /content, Kaggle in /kaggle/working
WORK = pathlib.Path("/content/bench" if os.path.isdir("/content") else "/kaggle/working/bench")
got = {}
for vid in IDS:
    f = WORK / vid / "video.mp4"
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        p = subprocess.run(["yt-dlp", "-q", "--no-warnings", "-f", "bv*[height<=1080][vcodec^=avc1]/bv*[height<=1080]",
                            "-o", str(f), f"https://www.youtube.com/watch?v={vid}"], capture_output=True, text=True)
        if p.returncode:
            print(vid, "download failed:", p.stderr.strip()[-300:])
    got[vid] = f.exists()
print("downloaded:", got)

In [ ]:
# Only if downloads failed (Colab only): copy the videos from Google Drive
if not all(got.values()) and os.path.isdir("/content"):
    from google.colab import drive
    drive.mount("/content/drive")
    for vid in IDS:
        src = pathlib.Path("/content/drive/MyDrive/vn-bench") / vid / "video.mp4"
        if not got[vid] and src.exists():
            shutil.copy(src, WORK / vid / "video.mp4")
            got[vid] = True
    print("available:", got)

## 3b. Is RapidOCR really on the GPU?

In [ ]:
import time, importlib.metadata as md
from rapidocr import RapidOCR, EngineType
print("rapidocr", md.version("rapidocr"), "| torch CUDA:", torch.cuda.is_available())
!ffmpeg -loglevel error -y -ss 290 -i {WORK}/BxV14h0kFs0/video.mp4 -frames:v 1 {WORK}/t.jpg
def bench(label, params):
    e = RapidOCR(params=params)
    e(str(WORK / "t.jpg"), use_cls=False)                      # warm-up
    t = time.time(); n = 0
    for _ in range(5):
        n = len(e(str(WORK / "t.jpg"), use_cls=False).txts or [])
    print(f"{label:14s} {(time.time() - t) / 5:.2f} s/frame, {n} lines")
    return e
if not torch.cuda.is_available():
    print("No GPU in this session: skipping the GPU check (frames.py will use RapidOCR on the CPU)")
else:
  gpu = {f"{k}.engine_type": EngineType.TORCH for k in ("Det", "Cls", "Rec")}
  gpu["EngineConfig.torch.use_cuda"] = True
  e = bench("torch + cuda", gpu)
  print("torch CUDA memory allocated:", round(torch.cuda.memory_allocated() / 1e6), "MB")
  for name, obj in vars(e).items():
      sess = getattr(obj, "session", None) or getattr(obj, "model", None)
      if sess is not None:
          print(" ", name, type(obj).__name__, getattr(sess, "device", ""), str(next(iter(getattr(sess, "parameters", lambda: iter([None]))()), "")).split("device=")[-1][:20] if hasattr(sess, "parameters") else "")
  bench("onnx cpu", {})
  !nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv
if not torch.cuda.is_available():
    bench("onnx cpu", {})


## 4. Run the three OCR modes

In [ ]:
MODES = ["hybrid", "rapidocr"] if torch.cuda.is_available() else ["hybrid"]  # full RapidOCR needs a GPU   # hybrid first: it's the answer that matters; tesseract alone = hybrid minus rescue
# Mode by mode, video by video. Each finished run is saved as frames.json and reused on a re-run,
# so a lost browser connection loses nothing: just run this cell again.
log = open(WORK / "run.log", "a")
for mode in MODES:
    for vid in [v for v in IDS if got[v]]:
        out = WORK / vid / f"frames_{mode}"
        if (out / "frames.json").exists():
            print(vid, mode, "already done", flush=True)
            continue
        p = subprocess.run(["python", "video-notes/skills/video-notes/scripts/frames.py", str(WORK / vid / "video.mp4"),
                            "--out", str(out), "--ocr", mode], capture_output=True, text=True)
        line = (p.stdout.strip().splitlines() or [""])[-1] if p.returncode == 0 else "ERROR " + (p.stderr or p.stdout)[-500:]
        print(vid, mode, line[:300], flush=True)
        log.write(f"{vid} {mode} {line}
"); log.flush()


## 5. Score against the hand labels

In [ ]:
scores = {}
for mode in MODES:
    s = subprocess.run(["python", "video-notes/bench/score.py", str(WORK), "--manifest", f"frames_{mode}/frames.json"],
                       capture_output=True, text=True).stdout
    scores[mode] = s
    print("==", mode); print(s)

## 6. Summary (copy this back)

In [ ]:
# Built from the saved manifests, so it works even after a reconnect
runs = {}
for f in sorted(WORK.glob("*/frames_*/frames.json")):
    m = json.loads(f.read_text())
    runs[f"{f.parent.parent.name}|{f.parent.name[7:]}"] = {"secs": m["secs"], "device": m["ocr"].get("device"),
        "rescued": m["ocr"].get("rescued", []), "kept": len(m["kept"]), "ocr_frames": m["ocr"]["frames"]}
summary = {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "cpus": __import__("os").cpu_count(), "runs": runs, "scores": scores}
print("VN-BENCH-RESULT " + json.dumps(summary))
